# Лабораторная №5 — шумоподавление с помощью автоэнкодера

Все семь этапов методички: загрузка MNIST, предобработка, визуализация чистых/зашумлённых изображений, топология автоэнкодера, обучение, тестирование, результат. Все пропуски кода заполнены.

## 1. Загрузка MNIST

In [ ]:
import numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
tf.keras.utils.set_random_seed(42)
(x_train,_),(x_test,_)=mnist.load_data()
print(x_train.shape,x_test.shape)

## 2. Предобработка — нормализация и гауссов шум

Автоэнкодер обучается на парах «зашумлённый вход → чистое изображение». Метки цифр не используются.

In [ ]:
x_train=x_train.astype("float32")/255.
x_test=x_test.astype("float32")/255.
x_train=np.expand_dims(x_train,-1)
x_test=np.expand_dims(x_test,-1)
rng=np.random.default_rng(42)
NOISE_FACTOR=0.5
x_train_noisy=np.clip(x_train+NOISE_FACTOR*rng.normal(size=x_train.shape),0.,1.).astype("float32")
x_test_noisy=np.clip(x_test+NOISE_FACTOR*rng.normal(size=x_test.shape),0.,1.).astype("float32")
print(x_train_noisy.shape,x_test_noisy.shape)

## 3. Вывод чистых и зашумлённых изображений

In [ ]:
fig,axs=plt.subplots(2,10,figsize=(16,4))
for i in range(10):
    axs[0,i].imshow(x_test[i,:,:,0],cmap="gray");axs[0,i].axis("off")
    axs[1,i].imshow(x_test_noisy[i,:,:,0],cmap="gray");axs[1,i].axis("off")
axs[0,0].set_title("Чистые");axs[1,0].set_title("Шум")
plt.tight_layout();plt.show()

## 4. Архитектура классического свёрточного автоэнкодера — допишите код

Энкодер сжимает изображение через Conv2D и MaxPooling2D; декодер восстанавливает размер через UpSampling2D. Выход sigmoid ограничивает пиксели диапазоном [0,1].

In [ ]:
inp=keras.Input(shape=(28,28,1))
x=layers.Conv2D(32,3,activation="relu",padding="same")(inp)
x=layers.MaxPooling2D(2,padding="same")(x)
x=layers.Conv2D(16,3,activation="relu",padding="same")(x)
encoded=layers.MaxPooling2D(2,padding="same")(x)
x=layers.Conv2D(16,3,activation="relu",padding="same")(encoded)
x=layers.UpSampling2D(2)(x)
x=layers.Conv2D(32,3,activation="relu",padding="same")(x)
x=layers.UpSampling2D(2)(x)
decoded=layers.Conv2D(1,3,activation="sigmoid",padding="same")(x)
autoencoder=keras.Model(inp,decoded)
autoencoder.compile(optimizer="adam",loss="binary_crossentropy",metrics=["mae"])
autoencoder.summary()

## 5. Обучение автоэнкодера — допишите код

In [ ]:
EPOCHS=10
history=autoencoder.fit(x_train_noisy,x_train,epochs=EPOCHS,batch_size=128,shuffle=True,validation_data=(x_test_noisy,x_test),verbose=1)

## 6. Тестирование — допишите код

In [ ]:
results=autoencoder.evaluate(x_test_noisy,x_test,verbose=0,return_dict=True)
print("Тестовые метрики:",results)

## 7. Результат шумоподавления — допишите код

In [ ]:
decoded_imgs=autoencoder.predict(x_test_noisy[:10],verbose=0)
fig,axs=plt.subplots(3,10,figsize=(16,5))
for i in range(10):
    for row,im in enumerate([x_test[i,:,:,0],x_test_noisy[i,:,:,0],decoded_imgs[i,:,:,0]]):
        axs[row,i].imshow(im,cmap="gray",vmin=0,vmax=1)
        axs[row,i].axis("off")
for row,title in enumerate(["Исходные","С шумом","Восстановленные"]):
    axs[row,0].set_ylabel(title)
plt.tight_layout();plt.show()

## 7.1. Графики обучения и сравнение ошибки

In [ ]:
plt.plot(history.history["loss"],label="train")
plt.plot(history.history["val_loss"],label="validation")
plt.xlabel("Эпоха");plt.ylabel("BCE");plt.legend();plt.show()
noisy_mse=np.mean((x_test_noisy-x_test)**2)
restored=autoencoder.predict(x_test_noisy[:1000],verbose=0)
restored_mse=np.mean((restored-x_test[:1000])**2)
print("MSE зашумлённых:",float(noisy_mse),"MSE восстановленных (1000 изображений):",float(restored_mse))

## Вывод

После выполнения сравните изображения, величины ошибки до и после шумоподавления и поведение train/validation. Фактические результаты появятся после запуска.